In [1]:
import os
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import celloracle as co
from sklearn.preprocessing import StandardScaler

# 设置绘图参数
plt.rcParams['figure.figsize'] = [6, 4]
sc.settings.verbosity = 3
sc.logging.print_header()
sc.settings.set_figure_params(dpi=80, facecolor='white')

# 加载单细胞RNA测序数据
# 假设数据已经是AnnData格式，包含原始counts矩阵
adata = sc.read_h5ad("/disk1/cai029/Caufussion/Benchmark_driver/Dataset/all_data.h5ad")
# 基本数据预处理
print(f"数据形状: {adata.shape}")
print(f"观察值（细胞）: {adata.n_obs}, 变量（基因）: {adata.n_vars}")

# 检查数据是否包含必要的元数据
print("可用的观察列:", adata.obs.columns.tolist())
print("可用的降维信息:", list(adata.obsm.keys()))

/disk1/cai029/anaconda3/envs/celloracle_env/lib/python3.10/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


数据形状: (22599, 20718)
观察值（细胞）: 22599, 变量（基因）: 20718
可用的观察列: ['NumberOfReads', 'AlignedToGenome', 'AlignedToTranscriptome', 'TranscriptomeUMIs', 'NumberOfGenes', 'CyclingScore', 'CyclingBinary', 'MutTranscripts', 'WtTranscripts', 'PredictionRF2', 'PredictionRefined', 'CellType', 'Score_HSC', 'Score_Prog', 'Score_GMP', 'Score_ProMono', 'Score_Mono', 'Score_cDC', 'Score_pDC', 'Score_earlyEry', 'Score_lateEry', 'Score_ProB', 'Score_B', 'Score_Plasma', 'Score_T', 'Score_CTL', 'Score_NK', 'sample_label', 'batch', 'NanoporeTranscripts', 'batch_origin', 'n_genes', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'stage']
可用的降维信息: ['X_pca', 'X_pca_harmony']


In [2]:
# # 质量控制 - 过滤低质量细胞和基因
# sc.pp.filter_cells(adata, min_genes=200)  # 至少表达200个基因的细胞
# sc.pp.filter_genes(adata, min_cells=10)    # 至少在3个细胞中表达的基因

# # 归一化处理
# sc.pp.normalize_total(adata, target_sum=1e4)
# sc.pp.log1p(adata)

# # 识别高变基因
sc.pp.highly_variable_genes(adata, n_top_genes=3000)
adata = adata[:, adata.var.highly_variable]

# 降维和聚类
#sc.pp.scale(adata, max_value=10)
sc.tl.pca(adata, svd_solver='arpack')
# sc.external.pp.harmony_integrate(adata, key='batch', basis='X_pca', max_iter_harmony=20)
sc.pp.neighbors(adata, n_pcs=30, n_neighbors=30,use_rep='X_pca_harmony')
sc.tl.umap(adata)
sc.tl.leiden(adata, resolution=0.6)  # 细胞聚类
sc.pl.umap(adata, color=['CellType'], legend_loc='on data')

extracting highly variable genes
    finished (0:00:00)
--> added
    'highly_variable', boolean vector (adata.var)
    'means', float vector (adata.var)
    'dispersions', float vector (adata.var)
    'dispersions_norm', float vector (adata.var)
computing PCA
    with n_comps=50
    finished (0:00:00)
computing neighbors
    finished: added to `.uns['neighbors']`
    `.obsp['distances']`, distances for each pair of neighbors
    `.obsp['connectivities']`, weighted adjacency matrix (0:00:14)
computing UMAP
    finished: added
    'X_umap', UMAP coordinates (adata.obsm)
    'umap', UMAP parameters (adata.uns) (0:00:07)
running Leiden clustering
    finished: found 9 clusters and added
    'leiden', the cluster labels (adata.obs, categorical) (0:00:01)


In [3]:
import networkx as nx
from scipy.sparse import issparse
from scipy.stats import spearmanr
import scipy.sparse as sparse
def prepare_network(adata, prior_net, add_edges_pct=0.001, corr_cutoff=0.6, uppercase=True):
    # Ensure gene names are uppercase to match network data
    adata.var_names_make_unique()
    if uppercase:
        adata.var_names = adata.var_names.str.upper()
        prior_net['from'] = prior_net['from'].str.upper()
        prior_net['to'] = prior_net['to'].str.upper()

    # Filter the network to include only genes present in adata
    net_filtered = prior_net[prior_net['from'].isin(adata.var_names) & prior_net['to'].isin(adata.var_names)]
    net_filtered = net_filtered.drop_duplicates(subset=['from', 'to'])

    # Create a directed graph using networkx
    network = nx.from_pandas_edgelist(net_filtered, 'from', 'to', create_using=nx.DiGraph)

    # Only keep the genes that exist in both single cell data and the prior gene interaction network
    network_nodes = list(network.nodes())
    adata_filtered = adata[:, network_nodes]

    # Calculate expression correlation to add additional edges
    gene_exp = pd.DataFrame(adata_filtered.X.toarray() if sparse.issparse(adata_filtered.X) else adata_filtered.X,
                            columns=adata_filtered.var_names)
    corr_matrix = gene_exp.corr(method='spearman').abs()
    np.fill_diagonal(corr_matrix.values, 0)

    # Identify additional edges based on correlation threshold
    additional_edges = corr_matrix.stack().rename_axis(['from', 'to']).reset_index(name='weight')
    additional_edges = additional_edges[additional_edges['weight'] > corr_cutoff]

    # Limit the number of additional edges based on the percentage
    top_k = int(len(network.nodes()) * (len(network.nodes()) - 1) / 2 * add_edges_pct)
    additional_edges = additional_edges.nlargest(top_k, 'weight')[['from', 'to']]

    # Add additional edges to the network, ensuring no duplicates
    existing_edges = set(network.edges())
    new_edges = [(row['from'], row['to']) for _, row in additional_edges.iterrows() if
                 (row['from'], row['to']) not in existing_edges]
    print(f"Adding {len(new_edges)} additional edges to the network")
    network.add_edges_from(new_edges)

    return network


def get_network(adata, network, keep_self_loops=True, average_degree=10, weight_degree=True):
    """
    Further process a gene regulatory network by calculating edge weights based on Spearman correlation
    coefficients from gene expression data, removing self-loops, adjusting edge weights based on node degrees,
    filtering edges to meet an average degree threshold, and finding the largest connected subgraph.
    """
    from scipy.sparse import issparse
    from scipy.stats import spearmanr
    # Convert adata to DataFrame for easier manipulation
    gene_exp = pd.DataFrame(adata.X.toarray() if issparse(adata.X) else adata.X, columns=adata.var_names)
    # Calculate Spearman correlation coefficient for all edges in the network
    for u, v in network.edges():
        if u in gene_exp.columns and v in gene_exp.columns:
            coef, _ = spearmanr(gene_exp[u], gene_exp[v])
            network[u][v]['weight'] = abs(coef)
    print("part1 complete")
    # Remove self-loops if not allowed
    if not keep_self_loops:
        network.remove_edges_from(nx.selfloop_edges(network))

    # Adjust edge weights based on the degree of connected nodes
    for u, v, d in network.edges(data=True):
        u_degree = network.degree(u)
        v_degree = network.degree(v)
        # Adjust the weight by the average degree of the two nodes
        if weight_degree:
            d['weight'] *= (u_degree + v_degree) / 2
        # print(u, v, d['weight'])
    print("part2 complete")
    # If average_degree is specified, select edges to keep based on their adjusted weights
    if average_degree is not None:
        # Calculate the total number of edges to keep based on the average_degree
        total_edges_to_keep = int(average_degree * len(network.nodes()))
        # Sort edges by their adjusted weight in descending order and keep the top ones
        all_edges_sorted_by_weight = sorted(network.edges(data=True), key=lambda x: x[2]['weight'], reverse=True)
        edges_to_keep = all_edges_sorted_by_weight[:total_edges_to_keep]

        # Create a new graph with the selected edges and their attributes
        network_filtered = nx.DiGraph()
        for u, v, d in edges_to_keep:
            network_filtered.add_edge(u, v, **d)
        network = network_filtered
    print("part2 complete")
    # Find the largest connected subgraph
    largest_components = max(nx.weakly_connected_components(network), key=len)
    if (len(largest_components) / len(network.nodes())) < 0.5:
        print('Warning: the size of the maximal connected subgraph is less than half of the input whole graph!')
    print("part3 complete")
    network = network.subgraph(largest_components).copy()

    # Save the network to CSV if a save path is provided

    print(f"Number of nodes: {len(network.nodes)}")
    print(f"Number of edges: {len(network.edges)}")
    return network

def network_master_regulators(prior_network,adata=None, topK=50, add_edges_pct=0.01, corr_cutoff=0.6,
                                  uppercase=False, net_degree=10, weight_degree=True, save_net_path=None, out_lam=0.8, driver_union=True,
                                  ILP_lam=0.5, **kwargs):
    network = prepare_network(adata, prior_network, add_edges_pct=add_edges_pct, corr_cutoff=corr_cutoff, uppercase=uppercase)
    print("prepare_network complete!")
    network = get_network(adata, network, average_degree=net_degree, weight_degree=weight_degree)
    print("get_network complete!")
    return network 

In [4]:
prior_net = pd.read_csv("/disk1/cai029/biosoft/CauFinder-main/resources/network/NicheNet_human.csv")
net_work = network_master_regulators(prior_network= prior_net,adata=adata)

Adding 69 additional edges to the network
prepare_network complete!
part1 complete
part2 complete
part2 complete
part3 complete
Number of nodes: 2537
Number of edges: 27510
get_network complete!


In [5]:
df_edges = nx.to_pandas_edgelist(net_work)
df_edges = df_edges[df_edges['source'] != df_edges['target']]
print(df_edges)

       source   target      weight
1         MYC   POU4F1  337.656548
2         MYC  RUNX1T1  290.442563
3         MYC     CAV1  277.284261
4         MYC    GATA2  265.566475
5         MYC      VIM  260.626321
...       ...      ...         ...
27505  COL6A2   COL1A2   12.304442
27506  COL6A2    LAMA4   10.406210
27507    RGS7      FOS   13.204857
27508   GNRH1      FOS   12.260766
27509   ENPEP    VCAM1   11.159542

[27383 rows x 3 columns]


In [6]:
# 使用扩散映射或PAGA推断发育轨迹
sc.tl.diffmap(adata)  # 扩散映射
sc.tl.paga(adata)     # PAGA图
sc.pl.paga(adata, plot=False)  # 可视化PAGA结果

# 选择根细胞（通常是最不成熟的细胞类型或健康对照）
# 这里假设簇0是起始点，根据您的数据调整
root_cell = np.where(adata.obs['CellType'] == 'HSC')[0][0]

# 计算扩散伪时间
adata.uns['iroot'] = root_cell
sc.tl.dpt(adata)  # 扩散伪时间

# 可视化伪时间
sc.pl.umap(adata, color=['dpt_pseudotime'], cmap='viridis')
plt.savefig("./1.png")

computing Diffusion Maps using n_comps=15(=n_dcs)
computing transitions
    finished (0:00:00)
    eigenvalues of transition matrix
    [1.         0.99146    0.98126304 0.9777129  0.9671842  0.95723116
     0.94598854 0.93412745 0.93005604 0.9194282  0.89880025 0.8902121
     0.8845486  0.87921613 0.86714876]
    finished: added
    'X_diffmap', diffmap coordinates (adata.obsm)
    'diffmap_evals', eigenvalues of transition matrix (adata.uns) (0:00:00)
running PAGA
    finished: added
    'paga/connectivities', connectivities adjacency (adata.uns)
    'paga/connectivities_tree', connectivities subtree (adata.uns) (0:00:00)
--> added 'pos', the PAGA positions (adata.uns['paga'])
computing Diffusion Pseudotime using n_dcs=10
    finished: added
    'dpt_pseudotime', the pseudotime (adata.obs) (0:00:00)


In [7]:
# net_work = df_edges.rename(columns={'source': 'TF', 'weight': 'importance'})
# 1. 创建pivot表,将source-target转换为矩阵格式  
base_grn = df_edges.pivot_table(  
    index='target',  # 靶基因作为行  
    columns='source',  # TF作为列  
    values='weight',  # 使用weight值  
    fill_value=0  # 没有连接的填充0  
)  
  
# 2. 转换为二值矩阵(如果需要)  
# CellOracle通常使用0/1表示是否存在调控关系  
# 您可以设置阈值将weight转换为0/1  
threshold = 10  # 根据您的数据调整  
base_grn_binary = (base_grn > threshold).astype(int)  
  
# 3. 添加必需的列  
base_grn_binary = base_grn_binary.reset_index()  
base_grn_binary.insert(0, 'peak_id', 'custom_peak_' + base_grn_binary.index.astype(str))  
base_grn_binary = base_grn_binary.rename(columns={'target': 'gene_short_name'})

In [8]:
# 创建Oracle对象并导入数据
oracle = co.Oracle()
oracle.import_anndata_as_raw_count(adata=adata,
                                  cluster_column_name="CellType",  # 聚类列名
                                  embedding_name="X_pca_harmony")       # 降维坐标名称

# 导入TF信息
oracle.import_TF_data(TF_info_matrix=base_grn_binary)

# KNN插值改善数据连续性
oracle.perform_PCA()
n_comps = min(np.where(np.diff(np.diff(np.cumsum(oracle.pca.explained_variance_ratio_)) > 0.002))[0][0], 50)
k = int(0.025 * oracle.adata.shape[0])  # 自动选择k值
oracle.knn_imputation(n_pca_dims=n_comps, k=k, balanced=True, 
                      b_sight=k*8, b_maxl=k*4, n_jobs=4)

# 构建细胞类型特异性GRN
links = oracle.get_links(cluster_name_for_GRN_unit="CellType", 
                         alpha=10, verbose_level=1)

# 过滤不可靠的连接
links.filter_links()

  0%|          | 0/15 [00:00<?, ?it/s]

In [9]:
# # 加载基础GRN（这里使用内置的人类启动子base GRN）
# base_GRN = co.data.load_human_promoter_base_GRN()
# print(f"基础GRN大小: {base_GRN.shape}")

# # 创建Oracle对象并导入数据
# oracle = co.Oracle()
# oracle.import_anndata_as_raw_count(adata=adata,
#                                   cluster_column_name="CellType",  # 聚类列名
#                                   embedding_name="X_pca_harmony")       # 降维坐标名称

# # 导入TF信息
# oracle.import_TF_data(TF_info_matrix=base_GRN)

# # KNN插值改善数据连续性
# oracle.perform_PCA()
# n_comps = min(np.where(np.diff(np.diff(np.cumsum(oracle.pca.explained_variance_ratio_)) > 0.002))[0][0], 50)
# k = int(0.025 * oracle.adata.shape[0])  # 自动选择k值
# oracle.knn_imputation(n_pca_dims=n_comps, k=k, balanced=True, 
#                       b_sight=k*8, b_maxl=k*4, n_jobs=4)

# # 构建细胞类型特异性GRN
# links = oracle.get_links(cluster_name_for_GRN_unit="CellType", 
#                          alpha=10, verbose_level=1)

# # 过滤不可靠的连接
# links.filter_links()

In [9]:
# 准备扰动模拟
oracle.get_cluster_specific_TFdict_from_Links(links_object=links)
oracle.fit_GRN_for_simulation(alpha=10, use_cluster_specific_TFdict=True)

# 获取所有转录因子列表
all_tfs_in_base_grn = base_grn_binary.columns[2:].tolist()  # 跳过'peak_id'和'gene_short_name'
print(f"基础GRN中定义的转录因子总数: {len(all_tfs_in_base_grn)}")

# 2. 从您的单细胞数据（adata）中获取所有实际检测到的基因
all_genes_in_expression_data = adata.var_names.tolist()
print(f"表达矩阵中检测到的基因总数: {len(all_genes_in_expression_data)}")

# 3. 取交集，得到可用于模拟的有效TF列表
available_tfs = list(set(all_tfs_in_base_grn) & set(all_genes_in_expression_data) & set(oracle.active_regulatory_genes))
available_tfs.sort()  # 排序方便查看
print(f"可用于扰动模拟的有效转录因子数量: {len(available_tfs)}")

  0%|          | 0/15 [00:00<?, ?it/s]

基础GRN中定义的转录因子总数: 847
表达矩阵中检测到的基因总数: 3000
可用于扰动模拟的有效转录因子数量: 739


In [10]:
# # 1. 在循环外执行一次性的、与TF无关的计算
# print("正在计算基础的细胞转移概率...")
#  # 显著调小参数
# print("基础计算完成。")

def reduce_embedding_dimension(oracle, target_dim=10):
    """降低嵌入维度以减少内存使用"""
    from sklearn.decomposition import PCA
    
    original_embedding = oracle.adata.obsm[oracle.embedding_name]
    
    # 使用PCA降维
    pca = PCA(n_components=target_dim)
    reduced_embedding = pca.fit_transform(original_embedding)
    
    # 更新嵌入
    oracle.adata.obsm[oracle.embedding_name] = reduced_embedding
    oracle.embedding = reduced_embedding
    
    print(f"嵌入维度从 {original_embedding.shape[1]} 降至 {target_dim}")
    return oracle

# 降低维度
print("降低嵌入维度...")
oracle_reduced = reduce_embedding_dimension(oracle, target_dim=10)

#随机选择一部分细胞进行分析  
import numpy as np  
np.random.seed(123)  
n_cells_subsample = 10000  
cell_idx_use = np.random.choice(len(oracle.adata), n_cells_subsample, replace=False)  

降低嵌入维度...
嵌入维度从 50 降至 10


In [11]:
# 计算扰动分数（与伪时间轨迹比较）
from celloracle.applications import Gradient_calculator, Oracle_development_module
import numpy as np  
if oracle.embedding.shape[1] != 2:  
    print(f"Warning: Oracle embedding has {oracle.embedding.shape[1]} dimensions")  
    print("Extracting first 2 dimensions or computing UMAP...")  
      
    # 选项A: 使用前两个维度  
    oracle.embedding = oracle.embedding[:, :2]  
    oracle.adata.obsm[oracle.embedding_name] = oracle.embedding.copy()  
# 计算发育梯度
gradient = Gradient_calculator(oracle_object=oracle, pseudotime_key="dpt_pseudotime")
n_grid = 40
gradient.calculate_p_mass(smooth=0.8, n_grid=n_grid, n_neighbors=200)
gradient.calculate_mass_filter(min_mass=7.4, plot=True)
gradient.transfer_data_into_grid(args={"method": "polynomial", "n_poly": 3})
gradient.calculate_gradient()

Extracting first 2 dimensions or computing UMAP...


In [12]:
from celloracle.applications import Oracle_development_module, Oracle_systematic_analysis_helper  
perturbation_results = {}
import time
for tf in available_tfs:  # 为演示只分析前20个TF，实际可分析全部
    try:  
        print("==========Use tf is :",tf,"=============")
        # 模拟TF敲低（表达量设为0）
        start_time = time.time()
        oracle.simulate_shift(perturb_condition={tf: 0.0}, n_propagation=3)
        oracle.estimate_transition_prob(n_neighbors=100,
                                        knn_random=True, 
                                        sampled_fraction=0.2, 
                                        calculate_randomized=True,
                                        cell_idx_use=cell_idx_use
                                       ) 
        oracle.calculate_embedding_shift(sigma_corr=0.05)
        perturbation_results[tf] = oracle.delta_embedding.copy()
        end_time = time.time()
        execution_time = end_time - start_time
        print(f"已完成 {tf} 的扰动模拟",f"执行时间：{execution_time} 秒")
        
    except ValueError as e:
        print(f"跳过 {tf}: {str(e)}")  

==========Use tf is : A2M =============
已完成 A2M 的扰动模拟 执行时间：58.9053475856781 秒
==========Use tf is : ABCA1 =============
已完成 ABCA1 的扰动模拟 执行时间：58.82108020782471 秒
==========Use tf is : ABCB1 =============
已完成 ABCB1 的扰动模拟 执行时间：59.404908895492554 秒
==========Use tf is : ABCC2 =============
已完成 ABCC2 的扰动模拟 执行时间：58.684650897979736 秒
==========Use tf is : ABI2 =============
已完成 ABI2 的扰动模拟 执行时间：59.578179359436035 秒
==========Use tf is : ABI3 =============
已完成 ABI3 的扰动模拟 执行时间：59.123034715652466 秒
==========Use tf is : ABLIM1 =============
已完成 ABLIM1 的扰动模拟 执行时间：59.29675579071045 秒
==========Use tf is : ACVR2A =============
已完成 ACVR2A 的扰动模拟 执行时间：59.341472864151 秒
==========Use tf is : ADM =============
已完成 ADM 的扰动模拟 执行时间：58.962583780288696 秒
==========Use tf is : AFAP1 =============
已完成 AFAP1 的扰动模拟 执行时间：60.08277654647827 秒
==========Use tf is : AFDN =============
已完成 AFDN 的扰动模拟 执行时间：58.97893667221069 秒
==========Use tf is : AGL =============
已完成 AGL 的扰动模拟 执行时间：59.134342432022095 秒
==========Use t

In [14]:
# 2. 为每个TF计算扰动分数
print("开始计算每个转录因子的扰动分数...")
perturbation_scores = {}

for tf, shift_data in perturbation_results.items():  
    print(f"正在处理转录因子: {tf}")
    dev = Oracle_development_module()  
    dev.load_differentiation_reference_data(gradient_object=gradient)  
    oracle.delta_embedding = shift_data  
    dev.load_perturb_simulation_data(oracle_object=oracle,   
                                     cell_idx_use=cell_idx_use,  
                                     name=f"{tf}_perturbation")  
    
    # 可选：诊断步骤，检查向量形状
    # 如果CellOracle版本允许，可以尝试打印内部向量维度进行确认
    # print(f"TF {tf} 的流程数据已加载。")
    
    # 计算内积分数  
    dev.calculate_inner_product()
    dev.calculate_digitized_ip(n_bins=10) 
    perturbation_scores[tf] = dev.inner_product_df['score'].sum()  
    print(f"  {tf} 的扰动分数: {perturbation_scores[tf]:.4f}")

# 3. 按扰动分数排序并输出结果  
print("\n关键TF排序（按扰动分数降序）:")  
sorted_tfs = sorted(perturbation_scores.items(), key=lambda x: x[1], reverse=True)  
for i, (tf, score) in enumerate(sorted_tfs[:10]):  
    print(f"{i+1}. {tf}: {score:.4f}")

开始计算每个转录因子的扰动分数...
正在处理转录因子: A2M
  A2M 的扰动分数: 0.1219
正在处理转录因子: ABCA1
  ABCA1 的扰动分数: -5.0281
正在处理转录因子: ABCB1
  ABCB1 的扰动分数: 7.4746
正在处理转录因子: ABCC2
  ABCC2 的扰动分数: 0.4381
正在处理转录因子: ABI2
  ABI2 的扰动分数: 11.0529
正在处理转录因子: ABI3
  ABI3 的扰动分数: -0.4841
正在处理转录因子: ABLIM1
  ABLIM1 的扰动分数: 2.0265
正在处理转录因子: ACVR2A
  ACVR2A 的扰动分数: 0.7314
正在处理转录因子: ADM
  ADM 的扰动分数: -1.7329
正在处理转录因子: AFAP1
  AFAP1 的扰动分数: 0.3212
正在处理转录因子: AFDN
  AFDN 的扰动分数: 0.3782
正在处理转录因子: AGL
  AGL 的扰动分数: 9.4780
正在处理转录因子: AHNAK
  AHNAK 的扰动分数: -36.9633
正在处理转录因子: AHSP
  AHSP 的扰动分数: -0.5645
正在处理转录因子: AIM2
  AIM2 的扰动分数: 0.1098
正在处理转录因子: AKAP12
  AKAP12 的扰动分数: 0.0911
正在处理转录因子: AKAP2
  AKAP2 的扰动分数: 4.1431
正在处理转录因子: ALAS2
  ALAS2 的扰动分数: -0.2893
正在处理转录因子: ALMS1
  ALMS1 的扰动分数: 1.6785
正在处理转录因子: ALOX5
  ALOX5 的扰动分数: -10.4228
正在处理转录因子: AMMECR1
  AMMECR1 的扰动分数: 0.3670
正在处理转录因子: ANKH
  ANKH 的扰动分数: -1.2139
正在处理转录因子: ANXA1
  ANXA1 的扰动分数: -23.3477
正在处理转录因子: APBB1
  APBB1 的扰动分数: 0.1530
正在处理转录因子: APC
  APC 的扰动分数: -0.6407
正在处理转录因子: APOC1
  APOC1 的扰动分数: 0.01

In [15]:
# 创建结果DataFrame
results_df = pd.DataFrame(sorted_tfs, columns=['TF', 'Perturbation_Score'])
results_df['Abs_Score'] = np.abs(results_df['Perturbation_Score'])
results_df = results_df.sort_values('Perturbation_Score', ascending=False)
results_df.to_csv("/disk1/cai029/Caufussion/Benchmark_driver/Methods/CellOracle/7_celloracle_key_biomarkers.csv")

In [17]:
# 可视化前20个关键TF
plt.figure(figsize=(12, 8))
top_tfs = results_df.head(20)
colors = ['red' if score < 0 else 'green' for score in top_tfs['Perturbation_Score']]

plt.barh(range(len(top_tfs)), top_tfs['Abs_Score'], color=colors, alpha=0.7)
plt.yticks(range(len(top_tfs)), top_tfs['TF'])
plt.xlabel('扰动分数绝对值')
plt.title('Perturbation')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig("/disk1/cai029/Caufussion/Benchmark_driver/Methods/CellOracle/8.png")
plt.show()

# 显示最重要的关键标志物
print("\n=== 疾病进展中最关键的前10个标志物 ===")
for i, row in results_df.head(10).iterrows():
    effect = "促进疾病进展" if row['Perturbation_Score'] > 0 else "抑制疾病进展"
    print(f"{row['TF']}: 扰动分数 = {row['Perturbation_Score']:.4f} ({effect})")


=== 疾病进展中最关键的前10个标志物 ===
MYC: 扰动分数 = 58.3081 (促进疾病进展)
GATA2: 扰动分数 = 49.1241 (促进疾病进展)
TCF4: 扰动分数 = 40.0986 (促进疾病进展)
ZEB1: 扰动分数 = 28.4380 (促进疾病进展)
CD34: 扰动分数 = 23.3520 (促进疾病进展)
HIST1H4C: 扰动分数 = 20.6740 (促进疾病进展)
SSR4: 扰动分数 = 19.8673 (促进疾病进展)
RPS6: 扰动分数 = 19.8303 (促进疾病进展)
RPS3A: 扰动分数 = 18.1804 (促进疾病进展)
MPO: 扰动分数 = 18.1484 (促进疾病进展)


In [ ]:
results_df